# Example 9.3 / 2: Choose the Output Keys

Companion notebook for **9.3 Writing Tables** in the AMPL lab Documentation, section "Choose the Output Keys". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Choose the Output Keys

Every output row needs a stable identity: one key column for values indexed over one set, a
compound key for values indexed over pairs. Do not replace a business key with row order;
sorting, filtering or inserting source rows would then attach results to the wrong records.

When the rows are taken from the data specifications (a plain `[product, period]` key), AMPL
writes the union of their indexing sets, and all items must have the same tuple dimension. Here
`sold` exists for periods 1 and 2 but `stock` also for period 0:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD := {"bands", "coils"};
param sold {PROD, 1..2} := 10;    # sales in periods 1 and 2
param stock {PROD, 0..2} := 5;    # inventory, including the starting period 0

table Balance OUT "amplcsv" "balance.csv":
    [product, period], sold, stock;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.write_table("Balance")
print(open("balance.csv").read())

product,period,sold,stock
bands,1,10,5
bands,2,10,5
coils,1,10,5
coils,2,10,5
bands,0,,5
coils,0,,5



### Pure AMPL: model.run

The period-0 rows appear with `sold` left blank. Blank representation and acceptance are
handler-specific. An explicit set-to-key mapping, such as `{p in PROD, t in 1..2} -> [product, period]`,
defines the exact row domain instead.

Pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
load amplcsv.dll;
write table Balance;
display sold, stock;

:       sold stock    :=
bands 0    .    5
bands 1   10    5
bands 2   10    5
coils 0    .    5
coils 1   10    5
coils 2   10    5
;

